# Этап 3. Анализ данных и гипотезы: Titanic

В этой работе я сначала проверяю и очищаю таблицу, затем беру случайные 25% данных для анализа. Идеи, которые появляются на этой части, проверяю на остальных 75% пассажиров. Так выводы получаются более надёжными.


## 1. Загрузка данных


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# В репозитории CSV находится в папке data.
# В Google Colab, если файла нет, появится окно для его загрузки.
DATA_PATH = Path('data/titanic_train.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('titanic_train.csv')
if not DATA_PATH.exists():
    from google.colab import files
    print('Выберите файл titanic_train.csv')
    uploaded = files.upload()
    DATA_PATH = Path(next(iter(uploaded)))

df = pd.read_csv(DATA_PATH)
print(f'Загружено строк: {len(df)}')
df.head()


## 2. Простая очистка данных

Сначала убираю полные дубликаты и проверяю логические ошибки: возраст не может быть отрицательным, цена билета — меньше нуля, а `Survived` может быть только 0 или 1.

Пропуски обрабатываю так:

- `Age` заполняю медианным возрастом пассажиров того же пола и класса;
- `Embarked` заполняю самым частым местом посадки;
- у `Cabin` слишком много пропусков, поэтому не придумываю номера кают. Вместо этого создаю простой признак `HasCabin`: известна каюта или нет.


In [ ]:
before_rows = len(df)
df_clean = df.drop_duplicates().copy()

valid = (
    df_clean['Survived'].isin([0, 1])
    & df_clean['Pclass'].isin([1, 2, 3])
    & (df_clean['Age'].isna() | df_clean['Age'].between(0, 100))
    & (df_clean['Fare'] >= 0)
)
df_clean = df_clean.loc[valid].copy()

df_clean['Age'] = df_clean['Age'].fillna(
    df_clean.groupby(['Sex', 'Pclass'])['Age'].transform('median')
)
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])
df_clean['HasCabin'] = df_clean['Cabin'].notna().astype(int)
df_clean = df_clean.drop(columns='Cabin')

print(f'Строк до очистки: {before_rows}')
print(f'Строк после очистки: {len(df_clean)}')
print(f'Незаполненных значений после очистки: {df_clean.isna().sum().sum()}')


## 3. Выборка для первого анализа

Беру 25% строк случайно. `random_state=42` нужен, чтобы при повторном запуске получалась та же выборка. Остальные строки не трогаю до проверки гипотез.


In [ ]:
analysis_sample = df_clean.sample(frac=0.25, random_state=42)
remaining_data = df_clean.drop(index=analysis_sample.index)

print(f'Выборка для анализа: {len(analysis_sample)} строк ({len(analysis_sample) / len(df_clean):.0%})')
print(f'Оставшаяся часть: {len(remaining_data)} строк')


## 4. Графики и основные числа


In [ ]:
stats = analysis_sample[['Age', 'Fare']].agg(['mean', 'median']).T
stats.columns = ['Среднее', 'Медиана']
display(stats.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

analysis_sample.groupby('Sex')['Survived'].mean().plot(
    kind='bar', ax=axes[0], color=['#4c78a8', '#72b7b2']
)
axes[0].set_title('Доля выживших по полу')
axes[0].set_xlabel('Пол')
axes[0].set_ylabel('Доля выживших')
axes[0].set_ylim(0, 1)

analysis_sample['Fare'].plot(kind='hist', bins=25, ax=axes[1], color='#4c78a8', edgecolor='white')
axes[1].set_title('Распределение стоимости билета')
axes[1].set_xlabel('Стоимость билета')

plt.tight_layout()
plt.show()


## 5. Гипотеза 1: женщины выживали чаще мужчин

На первом графике видно, что доля выживших у женщин выше. Проверяю эту мысль на оставшихся данных, а не только на исходной выборке.


In [ ]:
sample_h1 = analysis_sample.groupby('Sex')['Survived'].mean()
test_h1 = remaining_data.groupby('Sex')['Survived'].mean()

display(pd.DataFrame({'Выборка 25%': sample_h1, 'Оставшаяся часть': test_h1}).round(3))
print('Гипотеза подтверждается:', test_h1['female'] > test_h1['male'])


## 6. Гипотеза 2: в первом классе выживали чаще, чем в третьем

У пассажиров первого класса условия размещения и доступ к шлюпкам могли быть лучше. Сравниваю долю выживших по классу.


In [ ]:
sample_h2 = analysis_sample.groupby('Pclass')['Survived'].mean()
test_h2 = remaining_data.groupby('Pclass')['Survived'].mean()

display(pd.DataFrame({'Выборка 25%': sample_h2, 'Оставшаяся часть': test_h2}).round(3))
print('Гипотеза подтверждается:', test_h2.loc[1] > test_h2.loc[3])


## 7. Гипотеза 3: пассажиры с более дорогим билетом выживали чаще

Цена билета часто связана с классом пассажира. Беру медианную цену из выборки как границу: билет дешевле или дороже этой границы. Затем применяю ту же границу к оставшимся данным.


In [ ]:
fare_border = analysis_sample['Fare'].median()

def fare_groups(data, border):
    groups = np.where(data['Fare'] > border, 'Дороже медианы', 'Не дороже медианы')
    return data.assign(FareGroup=groups).groupby('FareGroup')['Survived'].mean()

sample_h3 = fare_groups(analysis_sample, fare_border)
test_h3 = fare_groups(remaining_data, fare_border)

display(pd.DataFrame({'Выборка 25%': sample_h3, 'Оставшаяся часть': test_h3}).round(3))
print('Гипотеза подтверждается:', test_h3['Дороже медианы'] > test_h3['Не дороже медианы'])


## 8. Вывод

После очистки данные подходят для дальнейшей работы. На случайной выборке появились три простые гипотезы — про пол, класс и цену билета. Все они проверяются на отдельной, оставшейся части таблицы. В следующем блокноте данные будут подготовлены для будущей модели.
